In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

silver_table_name = "workspace.aml_fraud_project.transactions_silver"
df_ml = spark.table(silver_table_name)

indexer = StringIndexer(inputCol="type", outputCol="type_index")

feature_columns = [
    "step", "type_index", "amount", 
    "oldbalanceOrg", "newbalanceOrig", "errorBalanceOrig",
    "oldbalanceDest", "newbalanceDest", "errorBalanceDest",
    "transaction_count_so_far"
]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training Rows: {train_df.count()}")
print(f"Testing Rows: {test_df.count()}")

In [0]:
import os
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
registered_model_name = "workspace.aml_fraud_project.fraud_rf_model"

mlflow_temp_path = "/Volumes/workspace/aml_fraud_project/raw_data/mlflow_tmp"
dbutils.fs.mkdirs(mlflow_temp_path)
os.environ["MLFLOW_DFS_TMP"] = mlflow_temp_path

rf = RandomForestClassifier(featuresCol="features", labelCol="isFraud", numTrees=50, maxDepth=5)
pipeline = Pipeline(stages=[indexer, assembler, rf])

with mlflow.start_run(run_name="AML_Fraud_Detection_Run") as run:
    
    trained_model = pipeline.fit(train_df)
    
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    
    sample_input = train_df.limit(5)
    sample_prediction = trained_model.transform(sample_input)
    

    sample_prediction_clean = sample_prediction.drop("features", "probability", "rawPrediction")
    
    signature = infer_signature(
        model_input=sample_input, 
        model_output=sample_prediction_clean
    )
    
    mlflow.spark.log_model(
        spark_model=trained_model,
        artifact_path="model",
        registered_model_name=registered_model_name,
        signature=signature,
        input_example=sample_input.limit(1).toPandas() 
    )
    
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col


predictions = trained_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="isFraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_roc = evaluator.evaluate(predictions)

print(f"Model AUC-ROC Score: {auc_roc:.4f}")

with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("auc_roc", auc_roc)

print("\nSample Predictions:")
display(predictions.select("amount", "errorBalanceOrig", "isFraud", "prediction", "probability").filter(col("isFraud") == 1).limit(5))